# W7D1 — Vision Transformers and Self-Supervised Learning — Lab

**Week 7 · Day 1 · Vision, language and speech** · Lab

<a href="https://colab.research.google.com/github/0xRush/AIEP_Olo_student/blob/master/week_7_multimodal/labs/D1_vit_ssl/D1_vit_ssl_blank.ipynb" target="_blank">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **The big idea.** A Vision Transformer is the transformer you built in week 5, with one new
> step in front: it cuts the image into **patches** and treats each patch as a token. And the
> best image features today come from models trained **without any labels** — they learned by
> hiding parts of images from themselves and predicting them.

Two halves today. First you open a pretrained ViT and find every piece of the transformer in it:
the patches, the `[CLS]` token, the attention. Then you meet two self-supervised models — one that
fills in hidden patches (MAE) and one whose features are so good that a plain logistic regression
on top of them beats the week-4 fine-tune.

<div dir="rtl" align="right">

# W7D1 — محوّلات الرؤية والتعلّم الذاتي الإشراف

**الأسبوع السابع · اليوم الأول · الرؤية واللغة والكلام** · معمل

<a href="https://colab.research.google.com/github/0xRush/AIEP_Olo_student/blob/master/week_7_multimodal/labs/D1_vit_ssl/D1_vit_ssl_blank.ipynb" target="_blank">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **الفكرة الكبرى.** محوّل الرؤية (ViT) هو المحوّل الذي بنيته في الأسبوع الخامس، مع خطوة جديدة واحدة
> في أوّله: يقصّ الصورة إلى **رقع** ويعامل كل رقعة كأنها رمز. وأفضل تمثيلات الصور اليوم تأتي من نماذج
> دُرّبت **دون أي تسميات** — تعلّمت بإخفاء أجزاء من الصور عن نفسها ثم توقّعها.

نصفان اليوم. أولًا تفتح محوّل رؤية مُدرَّبًا مسبقًا وتجد فيه كل قطعة من المحوّل: الرقع، ورمز `[CLS]`،
والانتباه. ثم تقابل نموذجين ذاتيَّي الإشراف — أحدهما يملأ الرقع المخفية (MAE)، والآخر تمثيلاته جيّدة
إلى درجة أن انحدارًا لوجستيًا بسيطًا فوقها يتفوّق على الضبط الدقيق في الأسبوع الرابع.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

- Cut an image into 16×16 patches with one reshape, and check it against PyTorch.
- Show that a ViT's "patch embedding" is just a `Conv2d` whose stride equals its kernel size.
- Explain why a ViT's sequence is 197 tokens long for 196 patches.
- Draw what the `[CLS]` token attends to, and say what an attention map does and does not prove.
- Run a masked autoencoder and see how much of an image it can rebuild.
- Train a **linear probe** on frozen self-supervised features and compare it with a supervised model.

<div dir="rtl" align="right">

## أهداف التعلّم

- أن تقصّ صورة إلى رقع ١٦×١٦ بإعادة تشكيل واحدة، وتتحقّق منها مقابل PyTorch.
- أن تُظهر أن «تضمين الرقع» في ViT مجرّد `Conv2d` خطوته تساوي نواته.
- أن تشرح لماذا طول متتالية ViT ‏١٩٧ رمزًا لـ١٩٦ رقعة.
- أن ترسم ما ينتبه إليه رمز `[CLS]`، وتقول ما تُثبته خريطة الانتباه وما لا تُثبته.
- أن تُشغّل مُرمِّزًا ذاتيًا مُقنَّعًا (MAE) وترى كم يستطيع أن يُعيد بناءه من الصورة.
- أن تدرّب **فحصًا خطيًا** على تمثيلات ذاتية الإشراف مجمّدة وتقارنه بنموذج مُشرَف.

</div>

## About the models and data

**Models** (downloaded once, then cached):

| model | what it is | size |
|---|---|---|
| `google/vit-base-patch16-224` | ViT-Base trained **with labels** (ImageNet classes) | 346 MB |
| `facebook/vit-mae-base` | a masked autoencoder: trained to rebuild hidden patches, no labels | 448 MB |
| `facebook/dinov2-small` | DINOv2: self-supervised features, no labels | 88 MB |

**Data:**

- `small_image_5class` — the 400 images of week 4 (bus, cat, dog, pizza, zebra), 224×224. We
  reuse **the exact train/val split of W4D5**, so today's numbers compare directly with the
  fine-tune you trained there (0.925).
- `sample_photos` — a few scene photos with several objects, for the attention maps.

Nothing is trained with gradients today. The slowest cell extracts features for 400 images, about
a minute on a laptop CPU.

<div dir="rtl" align="right">

## عن النماذج والبيانات

**النماذج** (تُنزَّل مرّة ثم تُحفظ):

| النموذج | ما هو | الحجم |
|---|---|---|
| `google/vit-base-patch16-224` | ViT-Base مُدرَّب **بتسميات** (فئات ImageNet) | ‏٣٤٦ ميجابايت |
| `facebook/vit-mae-base` | مُرمِّز ذاتي مُقنَّع: دُرّب على إعادة بناء الرقع المخفية، بلا تسميات | ‏٤٤٨ ميجابايت |
| `facebook/dinov2-small` | DINOv2: تمثيلات ذاتية الإشراف، بلا تسميات | ‏٨٨ ميجابايت |

**البيانات:**

- `small_image_5class` — صور الأسبوع الرابع الأربعمئة (حافلة، قطّ، كلب، بيتزا، حمار وحشي)، ٢٢٤×٢٢٤.
  ونُعيد استعمال **تقسيم التدريب/التحقّق نفسه من W4D5**، فتُقارَن أرقام اليوم مباشرةً بالضبط الدقيق
  الذي درّبته هناك (‏٠٫٩٢٥).
- `sample_photos` — بضع صور لمشاهد فيها عدّة أجسام، لخرائط الانتباه.

لا شيء يُدرَّب بالتدرّجات اليوم. أبطأ خلية تستخرج تمثيلات ٤٠٠ صورة، في نحو دقيقة على معالج حاسوب محمول.

</div>

## Setup

Run this first. On Colab it installs the course package; locally it uses your checkout.

<div dir="rtl" align="right">

## الإعداد

شغّل هذه الخلية أولًا. على Colab تُثبّت حزمة المقرّر، ومحلّيًا تستعمل نسختك من المستودع.

</div>

In [ ]:
# === AIEP portable setup — works locally and on Google Colab ========================
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
ensure('transformers', 'torch', 'scikit-learn', 'matplotlib', 'pandas')
seed_everything(42)

from aiep.data import get_dataset_dir, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, report

import json
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image

torch.set_grad_enabled(False)          # nothing is trained with gradients today
np.set_printoptions(precision=3, suppress=True)

SEED = 42
IMAGE_SIZE, PATCH_SIZE = 224, 16
GRID = IMAGE_SIZE // PATCH_SIZE        # 14 patches across, 14 down

IMAGE_ROOT = get_dataset_dir("small_image_5class") / "images"
PATHS = sorted(IMAGE_ROOT.rglob("*.jpg"))
LABELS = np.array([p.parent.name for p in PATHS])
PHOTOS = sorted(get_dataset_dir("sample_photos").rglob("*.jpg"))

# W4D5's split, recorded when that lab ran: the same 320 training and 80 validation images.
W4D5 = json.loads(load_artefact("w4d5_folds.json").read_text(encoding="utf-8"))
TRAIN_INDEX, VAL_INDEX = np.array(W4D5["train_index"]), np.array(W4D5["val_index"])
W4D5_FINETUNE = W4D5["w4d5_val_accuracy"]["C: layer4 unfrozen, lr/10"]

print(f"{len(PATHS)} images, classes {sorted(set(LABELS.tolist()))} | {len(PHOTOS)} scene photos")
print(f"W4D5 split: {len(TRAIN_INDEX)} train / {len(VAL_INDEX)} val, its fine-tune scored {W4D5_FINETUNE}")
print(versions(), "| device:", device())

## Section 1 — Warm-up: patches, and a ViT at work  *(~10 min)*

Nothing to write here. Two demos:

1. A 4×4 "image" cut into four 2×2 patches by one reshape. Each patch becomes a row of 4 numbers:
   a token.
2. A pretrained ViT classifies a photo. Same model all afternoon; in Section 2 you open it up.

<div dir="rtl" align="right">

## القسم ١ — إحماء: الرقع، ومحوّل رؤية يعمل  *(≈١٠ دقائق)*

لا شيء تكتبه هنا. عرضان:

١. «صورة» ٤×٤ تُقصّ إلى أربع رقع ٢×٢ بإعادة تشكيل واحدة. كل رقعة تصير صفًّا من ٤ أعداد: رمزًا.
٢. محوّل رؤية مُدرَّب مسبقًا يصنّف صورة. النموذج نفسه طوال الظهيرة؛ وفي القسم ٢ تفتحه.

</div>

In [ ]:
TOY = np.array([[1, 2, 0, 1],
                [0, 3, 1, 0],
                [2, 1, 4, 2],
                [1, 0, 2, 3]])

# (4, 4) -> (2 patch-rows, 2 rows inside, 2 patch-columns, 2 columns inside)
# -> put the two patch axes first -> one row of 4 numbers per patch
TOY_PATCHES = TOY.reshape(2, 2, 2, 2).swapaxes(1, 2).reshape(4, 4)
for k, patch in enumerate(TOY_PATCHES, start=1):
    print(f"patch {k}: {patch.tolist()}")
print("\nfour patches = four tokens. A 224x224 photo in 16x16 patches gives 14 x 14 = 196 tokens.")

In [ ]:
from transformers import AutoImageProcessor, ViTForImageClassification

processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
vit = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224",
                                                attn_implementation="eager").eval()

photo = Image.open(PATHS[0]).convert("RGB")
logits = vit(**processor(images=photo, return_tensors="pt")).logits[0]
top = logits.softmax(-1).topk(5)
plt.imshow(photo); plt.axis("off"); plt.title(PATHS[0].parent.name); plt.show()
for p, i in zip(top.values, top.indices):
    print(f"{p:.3f}  {vit.config.id2label[int(i)]}")

## Section 2 — Inside a ViT  *(~40 min)*

First, given: `patchify`, the warm-up's reshape for a real 224×224 colour image. Then three short
tasks, each finding one piece of the transformer inside the model:

1. **The patch embedding** — find it in the model and see that it is a `Conv2d`.
2. **197 tokens** — run the encoder and split the `[CLS]` token from the patch tokens.
3. **The attention map** — what does `[CLS]` look at?

The section ends with a given look at the twelve heads of one layer.

<div dir="rtl" align="right">

## القسم ٢ — داخل محوّل الرؤية  *(≈٤٠ دقيقة)*

أولًا، معطاة: `patchify`، إعادة تشكيل الإحماء لصورة ملوّنة حقيقية ٢٢٤×٢٢٤. ثم ثلاث مهام قصيرة، كل واحدة
تجد قطعة من المحوّل داخل النموذج:

١. **تضمين الرقع** — جِده في النموذج وانظر كيف أنه `Conv2d`.
٢. **‏١٩٧ رمزًا** — شغّل المُرمِّز وافصل رمز `[CLS]` عن رموز الرقع.
٣. **خريطة الانتباه** — إلامَ ينظر `[CLS]`؟

ويُختم القسم بنظرة معطاة على الرؤوس الاثني عشر في طبقة واحدة.

</div>

### Given — `patchify`

Same idea as the warm-up, with a colour channel: an `(H, W, 3)` image becomes
`(number of patches, 16·16·3)`. Read the three lines, run the cell, and see that it agrees with
`nn.Unfold`, PyTorch's own patch cutter.

<div dir="rtl" align="right">

### معطاة — `patchify`

الفكرة نفسها كما في الإحماء، مع قناة الألوان: صورة `(H, W, 3)` تصير `(عدد الرقع، 16·16·3)`. اقرأ الأسطر
الثلاثة، وشغّل الخلية، وانظر كيف تتّفق مع `nn.Unfold`، قاطع الرقع الخاص بـPyTorch.

</div>

In [ ]:
def patchify(image, patch_size):
    """(H, W, C) image -> (n_patches, patch_size * patch_size * C), patches in reading order."""
    height, width, channels = image.shape
    rows, cols = height // patch_size, width // patch_size
    grid = image.reshape(rows, patch_size, cols, patch_size, channels)    # five axes
    return grid.transpose(0, 2, 1, 3, 4).reshape(rows * cols, patch_size * patch_size * channels)


image = np.asarray(Image.open(PATHS[0]).convert("RGB"), dtype=np.float32) / 255
PATCHES = patchify(image, PATCH_SIZE)
print("patches:", PATCHES.shape, "  = 196 patches of 16 x 16 x 3 numbers")

# PyTorch's version gives (1, C*p*p, n_patches) with the channel axis first; reorder it to ours.
unfolded = nn.Unfold(PATCH_SIZE, stride=PATCH_SIZE)(torch.from_numpy(image).permute(2, 0, 1)[None])[0]
unfolded = unfolded.T.reshape(-1, 3, PATCH_SIZE * PATCH_SIZE).permute(0, 2, 1).reshape(len(PATCHES), -1)
UNFOLD_AGREES = np.allclose(PATCHES, unfolded.numpy())
print("same as nn.Unfold:", UNFOLD_AGREES)

### Task 1 — The patch embedding is a `Conv2d`

A ViT turns each 768-number patch into a 768-number token with one learned matrix. In the model
that matrix is stored as a `Conv2d` with a 16×16 kernel that jumps 16 pixels at a time — so the
windows never overlap, and each one is exactly one patch. Find it and check.

<div dir="rtl" align="right">

### المهمّة ١ — تضمين الرقع هو `Conv2d`

يحوّل ViT كل رقعة من ٧٦٨ عددًا إلى رمز من ٧٦٨ عددًا بمصفوفة واحدة مُتعلَّمة. وفي النموذج تُخزَّن تلك
المصفوفة طبقةَ `Conv2d` بنواة ١٦×١٦ تقفز ١٦ بكسلًا كل مرّة — فلا تتداخل النوافذ أبدًا، وكل نافذة رقعة
واحدة بالضبط. جِدها وتحقّق.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) print(vit) and look for patch_embeddings: it is under vit.vit.embeddings.
# 2) A Conv2d has .kernel_size and .stride. Compare the two.
#
# ١) اطبع `vit` وابحث عن `patch_embeddings`: إنها تحت `vit.vit.embeddings`.
# ٢) لطبقة `Conv2d` خاصّيتا `.kernel_size` و`.stride`. قارن بينهما.
# ────────────────────────────────────────────────────────────────────

# TODO: Store the patch embedding layer in `projection`, and compare its kernel size with its stride.
# مهمة: خزّن طبقة تضمين الرقع في `projection`، وقارن حجم نواتها بخطوتها.

print(projection)
print("kernel", projection.kernel_size, "stride", projection.stride, "-> equal:", STRIDE_EQUALS_KERNEL)

### Task 2 — 197 tokens

Run the image through the encoder (`vit.vit`, the model without its classification head) and look
at `last_hidden_state`. There are 196 patches — so why 197 rows? Row 0 is the **`[CLS]` token**: an
extra learned token, added in front, whose output is what the classifier reads.

<div dir="rtl" align="right">

### المهمّة ٢ — ‏١٩٧ رمزًا

مرّر الصورة في المُرمِّز (`vit.vit`، النموذج دون رأس التصنيف) وانظر إلى `last_hidden_state`. هناك ١٩٦
رقعة — فلماذا ١٩٧ صفًّا؟ الصف صفر هو **رمز `[CLS]`**: رمز إضافي مُتعلَّم يُضاف في الأوّل، وخرجه هو ما
يقرؤه المصنِّف.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) inputs = processor(images=photo, return_tensors="pt"), then vit.vit(**inputs).
# 2) .last_hidden_state has shape (1, tokens, 768). Row 0 is [CLS]; rows 1: are the patches.
#
# ١) `inputs = processor(images=photo, return_tensors="pt")`، ثم `vit.vit(**inputs)`.
# ٢) شكل `.last_hidden_state` هو `(1, tokens, 768)`. الصف صفر هو `[CLS]`، والصفوف `1:` هي الرقع.
# ────────────────────────────────────────────────────────────────────

# TODO: Run the encoder on `photo`, then split its output into CLS_TOKEN (row 0) and PATCH_TOKENS (the rest).
# مهمة: شغّل المُرمِّز على `photo`، ثم قسّم خرجه إلى `CLS_TOKEN` (الصف صفر) و`PATCH_TOKENS` (البقية).

SEQUENCE_LENGTH = hidden.shape[1]
print("encoder output:", tuple(hidden.shape))
print("[CLS] token   :", tuple(CLS_TOKEN.shape))
print("patch tokens  :", tuple(PATCH_TOKENS.shape))
print(f"{SEQUENCE_LENGTH} = 196 patches + 1 [CLS]")

### Task 3 — What does `[CLS]` look at?

Ask the model for its attention weights (`output_attentions=True`). The last layer gives a
`(heads, 197, 197)` matrix per image. Average the 12 heads, take **row 0** (what `[CLS]` attends
to), drop the column for `[CLS]` itself, and fold the other 196 numbers back into the 14×14 grid.
Laid over the photo, it shows where the classifier's token gathered its information.

<div dir="rtl" align="right">

### المهمّة ٣ — إلامَ ينظر `[CLS]`؟

اطلب من النموذج أوزان انتباهه (`output_attentions=True`). تعطي الطبقة الأخيرة مصفوفة
`(heads, 197, 197)` لكل صورة. خذ متوسّط الرؤوس الاثني عشر، وخذ **الصف صفر** (ما ينتبه إليه `[CLS]`)،
واحذف عمود `[CLS]` نفسه، وأعِد طيّ الأعداد الـ١٩٦ الباقية في الشبكة ١٤×١٤. وفوق الصورة تُظهر من أين جمع
رمز المصنِّف معلوماته.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) out = vit.vit(**inputs, output_attentions=True); out.attentions[-1][0] is (12, 197, 197).
# 2) If head is None take .mean(0) over the heads, else take [head].
# 3) row = weights[0]; grid = row[1:].reshape(GRID, GRID).
#
# ١) `out = vit.vit(**inputs, output_attentions=True)`؛ و`out.attentions[-1][0]` شكله `(12, 197, 197)`.
# ٢) إن كان `head` يساوي `None` فخذ `.mean(0)` على الرؤوس، وإلا فخذ `[head]`.
# ٣) `row = weights[0]`؛ ثم `grid = row[1:].reshape(GRID, GRID)`.
# ────────────────────────────────────────────────────────────────────

def cls_attention(image, head=None):
    """The [CLS] row of the last layer's attention, as a 14x14 grid, plus the whole 197-long row."""
    out = vit.vit(**processor(images=image, return_tensors="pt"), output_attentions=True)
    # TODO: Average the heads (or pick one), take row 0, and reshape its last 196 numbers to (GRID, GRID).
    # مهمة: خذ متوسّط الرؤوس (أو اختر واحدًا)، وخذ الصف صفر، وأعِد تشكيل أعداده الـ١٩٦ الأخيرة إلى `(GRID, GRID)`.


def overlay(image, grid, ax, title):
    big = np.asarray(Image.fromarray(grid).resize(image.size, Image.BILINEAR))
    ax.imshow(image); ax.imshow(big, cmap="inferno", alpha=0.55)
    ax.set_title(title, fontsize=9); ax.axis("off")


ROWS = []
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
for ax, path in zip(axes, PHOTOS[:3]):
    scene = Image.open(path).convert("RGB")
    grid, row = cls_attention(scene)
    ROWS.append(row)
    overlay(scene, grid, ax, path.name)
fig.suptitle("[CLS] attention, last layer, average of 12 heads")
plt.show()
print("each row sums to:", [round(float(r.sum()), 4) for r in ROWS])

### Given — Twelve heads, twelve views

The cell below draws heads 0, 3, 6 and 9 separately for one photo, with your `cls_attention`. Some
spread out, some pick one object.

**A caution worth repeating from week 5:** an attention map shows where information was *routed*.
It is not an explanation of the decision — a model can attend to the dog and still get it wrong.

<div dir="rtl" align="right">

### معطاة — اثنا عشر رأسًا، اثنتا عشرة نظرة

ترسم الخلية أدناه الرؤوس ٠ و٣ و٦ و٩ كلًّا على حدة لصورة واحدة، بدالتك `cls_attention`. بعضها ينتشر،
وبعضها يختار جسمًا واحدًا.

**تنبيه يستحق التكرار من الأسبوع الخامس:** خريطة الانتباه تُظهر أين *وُجِّهت* المعلومات. وليست تفسيرًا
للقرار — فقد ينتبه النموذج إلى الكلب ثم يُخطئ.

</div>

In [ ]:
HEADS = [0, 3, 6, 9]
scene = Image.open(PHOTOS[0]).convert("RGB")
HEAD_PEAKS = {}

fig, axes = plt.subplots(1, len(HEADS), figsize=(13, 3.6))
for ax, h in zip(axes, HEADS):
    grid, _ = cls_attention(scene, head=h)
    HEAD_PEAKS[h] = float(grid.max())
    overlay(scene, grid, ax, f"head {h}: peak {grid.max():.3f}")
plt.show()
print("peak weight per head:", {h: round(v, 3) for h, v in HEAD_PEAKS.items()})

### Takeaway

A ViT is a transformer encoder with a patch cutter in front and one extra token. Nothing in it
knows it is looking at an image except the first layer.

<div dir="rtl" align="right">

### الخلاصة

محوّل الرؤية مُرمِّزُ محوّلٍ أمامه قاطع رقع، ومعه رمز إضافي واحد. ولا شيء فيه يعرف أنه ينظر إلى صورة
إلا الطبقة الأولى.

</div>

## Section 3 — Features without labels  *(~30 min)*

The ViT above learned from 14 million **labelled** images. Labels are expensive. Self-supervised
models learn from images alone, by giving themselves a puzzle — a **pretext task**:

1. **MAE** hides most of an image's patches and learns to paint them back.
2. **DINOv2** learns features that stay the same across different crops of one image.

Neither ever saw a label. First, given: watch the MAE rebuild hidden patches. Then the question that
matters, in two tasks: are DINOv2's features any good? You answer it with a **linear probe** — the
simplest classifier there is, trained on top of frozen features.

<div dir="rtl" align="right">

## القسم ٣ — تمثيلات بلا تسميات  *(≈٣٠ دقيقة)*

تعلّم المحوّل أعلاه من ١٤ مليون صورة **مُسمّاة**. والتسميات مكلفة. أما النماذج ذاتية الإشراف فتتعلّم من
الصور وحدها، بأن تعطي نفسها لغزًا — **مهمّة ذريعة** (pretext task):

١. **MAE** يُخفي معظم رقع الصورة ويتعلّم أن يرسمها من جديد.
٢. **DINOv2** يتعلّم تمثيلات تبقى نفسها عبر قصّات مختلفة للصورة الواحدة.

لم يرَ أيٌّ منهما تسمية قط. أولًا، معطاة: شاهد MAE يُعيد بناء الرقع المخفية. ثم السؤال المهم، في مهمّتين:
هل تمثيلات DINOv2 جيّدة؟ تجيب عنه بـ**فحص خطي** — أبسط مصنِّف ممكن، يُدرَّب فوق تمثيلات مجمّدة.

</div>

### Given — The pretext task: rebuild what is hidden

The MAE below hides a fraction of the 196 patches (the **mask ratio**), sees only the rest, and
predicts the hidden ones. The two cells run it at 25%, 75% and 90% and draw the result. At 75% — the
setting it was trained with — its encoder sees only 49 patches.

<div dir="rtl" align="right">

### معطاة — المهمّة الذريعة: أعِد بناء ما خفي

يُخفي MAE أدناه نسبةً من الرقع الـ١٩٦ (**نسبة الإخفاء**)، ولا يرى إلا الباقي، ويتوقّع المخفيّ. وتُشغّله
الخليتان عند ٢٥٪ و٧٥٪ و٩٠٪ وترسمان النتيجة. وعند ٧٥٪ — النسبة التي دُرّب بها — لا يرى مُرمِّزه إلا ٤٩ رقعة.

</div>

In [ ]:
from transformers import ViTMAEForPreTraining

mae_processor = AutoImageProcessor.from_pretrained("facebook/vit-mae-base")
mae = ViTMAEForPreTraining.from_pretrained("facebook/vit-mae-base").eval()
MAE_PIXELS = mae_processor(images=[Image.open(p).convert("RGB") for p in PATHS[::100]],
                           return_tensors="pt")["pixel_values"]          # 4 images, one per 100
MEAN = torch.tensor(mae_processor.image_mean).view(1, 3, 1, 1)
STD = torch.tensor(mae_processor.image_std).view(1, 3, 1, 1)


def set_ratio(ratio):
    """The mask ratio lives in the model's config."""
    mae.config.mask_ratio = ratio
    mae.vit.embeddings.config.mask_ratio = ratio


def reconstruct(ratio):
    """Hide `ratio` of the patches; return (the visible patches, visible + predicted patches)."""
    set_ratio(ratio)
    noise = torch.rand(len(MAE_PIXELS), 196, generator=torch.Generator().manual_seed(SEED))
    out = mae(pixel_values=MAE_PIXELS, noise=noise)
    truth = mae.patchify(MAE_PIXELS)
    hidden = out.mask.unsqueeze(-1)                          # 1 where the patch was hidden
    show = lambda patches: (mae.unpatchify(patches) * STD + MEAN).clamp(0, 1).permute(0, 2, 3, 1)
    return show(truth * (1 - hidden)), show(truth * (1 - hidden) + out.logits * hidden), out

In [ ]:
RATIOS = [0.25, 0.75, 0.90]
LOSSES = {}
fig, axes = plt.subplots(2, len(RATIOS), figsize=(10, 6.6))
for column, ratio in enumerate(RATIOS):
    masked, rebuilt, out = reconstruct(ratio)
    LOSSES[ratio] = float(out.loss)
    axes[0, column].imshow(masked[0]); axes[0, column].set_title(f"{ratio:.0%} hidden")
    axes[1, column].imshow(rebuilt[0]); axes[1, column].set_title(f"rebuilt — loss {out.loss:.3f}")
for ax in axes.ravel():
    ax.axis("off")
plt.show()

set_ratio(0.75)
VISIBLE_AT_75 = mae.vit(pixel_values=MAE_PIXELS[:1], noise=torch.rand(1, 196)).last_hidden_state.shape[1] - 1
print("loss on the hidden patches:", {r: round(v, 3) for r, v in LOSSES.items()})
print(f"at 75% the encoder saw {VISIBLE_AT_75} of 196 patches")

### Task 1 — A linear probe on frozen features

A **linear probe** asks: is the class already *in* the features, in a form a straight line can
separate? Freeze the model, turn each image into one vector, and fit a `LogisticRegression` on
W4D5's 320 training images. Score it on the same 80 validation images W4D5 used.

Do it twice: with **DINOv2** (no labels ever) and with the **supervised ViT** from Section 2.

<div dir="rtl" align="right">

### المهمّة ١ — فحص خطي على تمثيلات مجمّدة

يسأل **الفحص الخطي**: هل الفئة موجودة *في* التمثيلات أصلًا، بصورة يستطيع خطٌّ مستقيم أن يفصلها؟ جمّد
النموذج، وحوّل كل صورة إلى متّجه واحد، ودرّب `LogisticRegression` على صور التدريب الـ٣٢٠ من W4D5. ثم
قيّمه على صور التحقّق الـ٨٠ نفسها التي استعملها W4D5.

افعل ذلك مرّتين: بـ**DINOv2** (بلا تسميات قط) وبـ**ViT المُشرَف** من القسم ٢.

</div>

In [ ]:
from transformers import AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

dino_processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dino = AutoModel.from_pretrained("facebook/dinov2-small").eval()
for p in dino.parameters():
    p.requires_grad = False              # frozen: a probe never changes the backbone


def embed_all(encoder, image_processor, batch=25):
    """One vector per image: the encoder's [CLS] output, for all 400 images."""
    vectors = []
    for start in range(0, len(PATHS), batch):
        images = [Image.open(p).convert("RGB") for p in PATHS[start:start + batch]]
        pixels = image_processor(images=images, return_tensors="pt")["pixel_values"]
        vectors.append(encoder(pixel_values=pixels).last_hidden_state[:, 0].numpy())
    return np.concatenate(vectors)


started = time.perf_counter()
DINO_FEATURES = embed_all(dino, dino_processor)
VIT_FEATURES = embed_all(vit.vit, processor)
print(f"DINOv2 {DINO_FEATURES.shape}, ViT {VIT_FEATURES.shape} in {time.perf_counter() - started:.0f} s")

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
# 2) model.fit(features[TRAIN_INDEX], LABELS[TRAIN_INDEX])
# 3) return model.score(features[VAL_INDEX], LABELS[VAL_INDEX])
#
# ١) `model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))`
# ٢) `model.fit(features[TRAIN_INDEX], LABELS[TRAIN_INDEX])`
# ٣) `return model.score(features[VAL_INDEX], LABELS[VAL_INDEX])`
# ────────────────────────────────────────────────────────────────────

def linear_probe(features, train_index=TRAIN_INDEX):
    """Fit a logistic regression on the training rows; return its validation accuracy."""
    # TODO: Fit a scaled LogisticRegression on features[train_index] and return its accuracy on VAL_INDEX.
    # مهمة: درّب `LogisticRegression` مع توحيد المقياس على `features[train_index]` وأعِد دقّته على `VAL_INDEX`.


DINO_PROBE = linear_probe(DINO_FEATURES)
VIT_PROBE = linear_probe(VIT_FEATURES)
RESULTS = pd.DataFrame([
    ("DINOv2 features + linear probe", "none", DINO_PROBE),
    ("supervised ViT features + linear probe", "14M ImageNet labels", VIT_PROBE),
    ("W4D5 ResNet-18 fine-tune", "ImageNet + your 320", W4D5_FINETUNE),
], columns=["model", "labels used to pretrain", "val accuracy"])
print(RESULTS.to_string(index=False))

### Task 2 — How many labels do you need?

Self-supervised features matter most when labels are scarce. Train the DINOv2 probe on only 10,
25, 50 and 100 of the training images (and on all 320) and plot accuracy against label count.

<div dir="rtl" align="right">

### المهمّة ٢ — كم تسمية تحتاج؟

تكون التمثيلات ذاتية الإشراف أثمن ما تكون حين تندر التسميات. درّب فحص DINOv2 على ١٠ و٢٥ و٥٠ و١٠٠ فقط من
صور التدريب (وعلى الـ٣٢٠ كلها) وارسم الدقّة مقابل عدد التسميات.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) For each n in BUDGETS take subset = rng.choice(TRAIN_INDEX, n, replace=False).
# 2) Use your linear_probe(DINO_FEATURES, train_index=subset) and store it in CURVE[n].
#
# ١) لكل `n` في `BUDGETS` خذ `subset = rng.choice(TRAIN_INDEX, n, replace=False)`.
# ٢) استعمل `linear_probe(DINO_FEATURES, train_index=subset)` وخزّنه في `CURVE[n]`.
# ────────────────────────────────────────────────────────────────────

BUDGETS = [10, 25, 50, 100, len(TRAIN_INDEX)]
rng = np.random.default_rng(SEED)
CURVE = {}
# TODO: For every budget, probe DINOv2 on a random subset of that many training images; store CURVE[n].
# مهمة: لكل ميزانية، افحص DINOv2 على مجموعة عشوائية بذلك العدد من صور التدريب؛ وخزّن `CURVE[n]`.

plt.plot(list(CURVE), list(CURVE.values()), "o-", label="DINOv2 + linear probe")
plt.axhline(W4D5_FINETUNE, color="grey", ls="--", label=f"W4D5 fine-tune, 320 labels ({W4D5_FINETUNE})")
plt.xscale("log"); plt.xlabel("labelled training images"); plt.ylabel("val accuracy")
plt.ylim(0, 1.05); plt.legend(); plt.show()
print({n: round(a, 3) for n, a in CURVE.items()})

### Write-up

In **one sentence**: DINOv2 never saw a label, yet a straight line on its features matches a ViT
trained on 14 million labelled images and beats the model you fine-tuned in W4D5. Why can that happen?

> *Example: "The pretext task forced DINOv2 to learn what makes images different from each other
> across 140 million photos, and those differences already line up with our five classes — so the
> probe only has to draw the lines, not learn the features."*

<div dir="rtl" align="right">

### كتابة

في **جملة واحدة**: لم يرَ DINOv2 تسمية قط، ومع ذلك يُضاهي خطٌّ مستقيم على تمثيلاته محوّلَ رؤية دُرّب على ١٤
مليون صورة مُسمّاة، ويتفوّق على النموذج الذي ضبطته بدقّة في W4D5. كيف يمكن ذلك؟

> *مثال: «أجبرت المهمّة الذريعة DINOv2 على تعلّم ما يجعل الصور مختلفة بعضها عن بعض عبر ١٤٠ مليون صورة،
> وهذه الفروق تصطفّ أصلًا مع فئاتنا الخمس — فلا يبقى على الفحص إلا أن يرسم الخطوط، لا أن يتعلّم التمثيلات.»*

</div>

In [ ]:
YOUR_ANSWER = """
(write your sentence here)
"""

In [ ]:
(ARTEFACT_DIR / "probe_results.json").write_text(json.dumps({
    "dino_probe": DINO_PROBE, "vit_probe": VIT_PROBE, "w4d5_finetune": W4D5_FINETUNE,
    "label_curve": {str(n): a for n, a in CURVE.items()}}, indent=2), encoding="utf-8")
print("wrote probe_results.json — tomorrow's CLIP lab compares against it")

## Wrap-up

- A ViT = patch cutter (`Conv2d`, stride = kernel) + `[CLS]` + position embeddings + the
  transformer encoder from week 5.
- Attention maps show routing, not reasons.
- Self-supervised pretraining (MAE, DINOv2) learns from images alone, and its frozen features
  plus a linear probe match a model trained on 14M labels and beat a supervised fine-tune —
  with as few as 10 labels per class.

Tomorrow (**D2**) the features get a second modality: CLIP puts images and *text* in one space,
and a VLM lets a language model read the image tokens you met today.

<div dir="rtl" align="right">

## الخلاصة

- ViT = قاطع رقع (`Conv2d` خطوته تساوي نواته) + `[CLS]` + تضمينات الموضع + مُرمِّز المحوّل من الأسبوع
  الخامس.
- خرائط الانتباه تُظهر التوجيه، لا الأسباب.
- التدريب المسبق الذاتي الإشراف (MAE وDINOv2) يتعلّم من الصور وحدها، وتمثيلاته المجمّدة مع فحص خطي
  تُضاهي نموذجًا دُرّب على ١٤ مليون تسمية وتتفوّق على ضبط دقيق مُشرَف — بعشر تسميات فقط لكل فئة.

وغدًا (**اليوم ٢**) تكتسب التمثيلات وسيطًا ثانيًا: يضع CLIP الصور و*النصوص* في فضاء واحد، ويتيح نموذج
الرؤية واللغة (VLM) لنموذج لغوي أن يقرأ رموز الصورة التي قابلتها اليوم.

</div>

In [ ]:
check(PATCHES.shape == (196, 768) and UNFOLD_AGREES,
      f"patchify must give (196, 768) and match nn.Unfold — got {PATCHES.shape}, match {UNFOLD_AGREES}",
      f"يجب أن تُعطي `patchify` الشكل `(196, 768)` وتطابق `nn.Unfold` — والناتج {PATCHES.shape}، والتطابق {UNFOLD_AGREES}")
check(STRIDE_EQUALS_KERNEL and SEQUENCE_LENGTH == 197,
      f"the patch embedding must be a Conv2d with stride == kernel, and the sequence 197 long — got "
      f"{STRIDE_EQUALS_KERNEL} and {SEQUENCE_LENGTH}",
      f"يجب أن يكون تضمين الرقع `Conv2d` خطوته تساوي نواته، وطول المتتالية ١٩٧ — والناتج "
      f"{STRIDE_EQUALS_KERNEL} و{SEQUENCE_LENGTH}")
check(len(ROWS) == 3 and all(abs(r.sum() - 1) < 1e-4 for r in ROWS) and len(HEAD_PEAKS) == 4,
      "each [CLS] attention row must sum to 1 (average the heads, do not add them), and four heads must be drawn",
      "يجب أن يجمع كل صفّ انتباه لـ`[CLS]` إلى ١ (خذ متوسّط الرؤوس لا مجموعها)، وأن تُرسم أربعة رؤوس")
check(VISIBLE_AT_75 == 49 and LOSSES.get(0.25, 1) < LOSSES.get(0.90, 0),
      f"at 75% the MAE encoder must see 49 patches, and hiding more must cost more — got {VISIBLE_AT_75}, {LOSSES}",
      f"عند ٧٥٪ يجب أن يرى مُرمِّز MAE ‏٤٩ رقعة، وأن يكلّف إخفاء المزيد أكثر — والناتج {VISIBLE_AT_75}، {LOSSES}")
check(DINO_PROBE > W4D5_FINETUNE and all(not p.requires_grad for p in dino.parameters()),
      f"the frozen DINOv2 probe must beat W4D5's fine-tune ({W4D5_FINETUNE}) — got {DINO_PROBE:.4f}",
      f"يجب أن يتفوّق فحص DINOv2 المجمّد على ضبط W4D5 الدقيق ({W4D5_FINETUNE}) — والناتج {DINO_PROBE:.4f}")
check(len(CURVE) == len(BUDGETS) and CURVE[10] > 0.5,
      f"the label curve must have every budget, and 10 labels must already beat chance (0.2) by far — got {CURVE}",
      f"يجب أن يضمّ منحنى التسميات كل الميزانيات، وأن تتفوّق ١٠ تسميات على الصدفة (٠٫٢) بفارق كبير — والناتج {CURVE}")
report()